<a href="https://colab.research.google.com/github/likitreddy/Career-Guide-Agent/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import os
import json
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get('HF_TOKEN')
MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
client = InferenceClient(model=MODEL, token=HF_TOKEN)

In [37]:
def ask_hf(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    try:
        response = client.chat_completion(
            messages=messages,
            max_tokens=1024,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

In [38]:
def step1_career_recommendation(profile: dict) -> str:
    prompt = f"""
You are a career counselor AI. Based on this student profile, suggest the
TOP 3 most suitable career paths with a one-line reason for each.

Profile:
- Education: {profile['education']}
- Skills: {', '.join(profile['skills'])}
- Interests: {', '.join(profile['interests'])}

Respond in this format:
1. <Career> - <reason>
2. <Career> - <reason>
3. <Career> - <reason>
"""
    return ask_hf(prompt)

def step2_skill_gap_analysis(profile: dict, target_career: str) -> str:
    prompt = f"""
You are a skill-gap analysis AI. The student wants to pursue: {target_career}

Their current skills are: {', '.join(profile['skills'])}

List the TOP 5 missing/important skills they need to learn for this career,
ordered by priority (most important first). Keep each point short.
"""
    return ask_hf(prompt)

def step3_learning_roadmap(target_career: str, skill_gaps: str) -> str:
    prompt = f"""
You are a learning roadmap generator AI. The student wants to become a:
{target_career}

Their skill gaps are:
{skill_gaps}

Create a 3-month personalized learning roadmap (Month 1, Month 2, Month 3),
with 2-3 bullet points per month covering what to learn and one suggested
free resource type (e.g., course, project, certification) per skill.
"""
    return ask_hf(prompt)

In [39]:
def run_agent(profile: dict, target_career: str = None):
    print("\n=== STEP 1: Career Recommendations ===")
    careers = step1_career_recommendation(profile)
    print(careers)

    if "Error: 403" in careers:
        print("\nSTOPPED: Please check your HF_TOKEN permissions.")
        return

    if not target_career:
        target_career = input("\nEnter your chosen career from above: ")

    print(f"\n=== STEP 2: Skill Gap Analysis for {target_career} ===")
    gaps = step2_skill_gap_analysis(profile, target_career)
    print(gaps)

    print(f"\n=== STEP 3: 3-Month Learning Roadmap ===")
    roadmap = step3_learning_roadmap(target_career, gaps)
    print(roadmap)

    result = {
        "career_recommendations": careers,
        "target_career": target_career,
        "skill_gaps": gaps,
        "roadmap": roadmap,
    }

    with open("agent_output.json", "w") as f:
        json.dump(result, f, indent=2)
    print("\n✅ Results saved to agent_output.json")
    return result

In [ ]:
student_profile = {
    "education": "B.Tech in Information Technology, 2nd year",
    "skills": ["Python", "SQL", "NumPy", "Pandas", "Basic JavaScript"],
    "interests": ["Artificial Intelligence", "Data Science", "Web Development"],
}

run_agent(student_profile)


=== STEP 1: Career Recommendations ===
Error: (Request ID: Root=1-6a2d091b-73edbcb06bb78bb069e140fd;4ab0bf2a-ff81-4598-91ae-13f499caefaf)

403 Forbidden: This authentication method does not have sufficient permissions to call Inference Providers on behalf of user Likitreddy2007.
Cannot access content at: https://router.huggingface.co/v1/chat/completions.
Make sure your token has the correct permissions.
